# Speaker study: EXPLORATORY layer sweep, Gemma 2 2B base only

Not preregistered; plan in `DEVIATIONS.md` (entries 34-35), loading changes in entries 36-37. For every
decoder block loaded (all 26 blocks) it rebuilds the paper's S1/S2 pain vectors from the §3.1 sentences and
projects all 1,260 stimuli onto them. Forward passes only.

**Loading:** weights go straight onto the GPU (`device_map={"": 0}`), without the LM head and without a
full copy in CPU RAM; no offloading and no quantization. If any weight is not on the GPU after loading, the
run stops with an `ERROR:` line instead of continuing slowly on the CPU.

**How to run:** start from a fresh runtime (Runtime -> Disconnect and delete runtime), select a T4 GPU,
then Runtime -> Run all. about 10-15 min. The last cell downloads `speaker_study_layersweep_gemma.zip` (also saved to
`MyDrive/speaker_study/`); attach it in the Claude session.

In [ ]:
# 1. GPU check. Runtime -> Change runtime type -> T4 GPU (free tier).
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a GPU first."

In [ ]:
# 2. Mount Google Drive; all outputs go to MyDrive/speaker_study/ (never overwritten).
from google.colab import drive
drive.mount("/content/drive")
OUT_ROOT = "/content/drive/MyDrive/speaker_study"
import os; os.makedirs(OUT_ROOT, exist_ok=True)

In [ ]:
# 3. Hugging Face token from Colab secrets (key icon on the left, name HF_TOKEN, notebook access on).
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [ ]:
# 4. Clone the paper's repo at the pinned commit (read-only use; nothing in it is modified).
!rm -rf /content/Pain-axis
!git clone -q https://github.com/valen-research/Pain-axis /content/Pain-axis
!git -C /content/Pain-axis checkout -q 7c256502ed3d98e4e6379290fe7db2f93cb8d025
!git -C /content/Pain-axis log -1 --format='%H %s'
# Uses Colab's preinstalled torch/transformers/accelerate/pandas (versions are logged to env.txt).
!mkdir -p /content/speaker_study_scripts

In [ ]:
%%writefile /content/speaker_study_scripts/pa_common.py
"""Shared helpers for the speaker study: dataset and vector loading, model loading, and
final-token readout. The format, readout and projection logic is copied from
Pain-axis scripts/4.1_self_other/01_screen_scenarios.py (commit 7c25650) so that our
projections are comparable with the shipped ones.
"""

import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch

PAIN_AXIS_COMMIT = "7c256502ed3d98e4e6379290fe7db2f93cb8d025"
SEED = 0

# Dataset stratum -> our group name.
STRATUM_TO_GROUP = {
    "self_directed": "harm_to_model",
    "vicarious_empathic": "user_suffering",
    "neutral_filler": "neutral",
}

# Same keys and order as the repo's 4.1 screen.
VECTOR_KEYS = [
    "s1_pain_vector", "s2_pain_vector",
    "fear_vector", "negemotion_vector", "negworld_vector",
    "bodysens_vector", "arousal_vector", "random_vector", "numb_vector", "sadness_vector",
]

# Base models from the spec: HF repo -> repo's model name (used in vector/result file names).
MODELS = {
    "google/gemma-2-2b": "Gemma_2_2B_base",
    "Qwen/Qwen2.5-7B": "Qwen_2.5_7B_base",
    "meta-llama/Llama-3.1-8B": "Llama_3.1_8B_base",
    "google/gemma-2-9b": "Gemma_2_9B_base",
}

DTYPES = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}


def load_scenarios(pain_axis_dir):
    path = Path(pain_axis_dir) / "datasets" / "4.1_self_other_420_scenarios.json"
    with open(path, encoding="utf-8") as f:
        items = json.load(f)
    for it in items:
        it["group"] = STRATUM_TO_GROUP[it["stratum"]]
        it["n_user_turns"] = sum(line.startswith("[User]:") for line in it["text"].split("\n"))
    return items


def load_vectors(pain_axis_dir, model_name):
    """Unit vectors from the file the 4.1 screen uses, and its layer."""
    path = Path(pain_axis_dir) / "results" / "vectors_full_steering" / f"vectors_full_{model_name}.pt"
    data = torch.load(path, map_location="cpu", weights_only=False)
    units = {}
    for k in VECTOR_KEYS:
        if data.get(k) is not None:
            v = data[k].float().numpy()
            n = np.linalg.norm(v)
            units[k] = v / n if n > 0 else v
    return int(data["layer"]), units


def load_shipped_screen(pain_axis_dir, model_name):
    import pandas as pd
    path = Path(pain_axis_dir) / "results" / "4.1_self_other" / "per_model" / f"screen_v2_{model_name}.csv"
    return pd.read_csv(path)


def load_model(repo, dtype="bf16", attn="default", keep_layers=None):
    """Load tokenizer and model on the GPU.

    keep_layers: if set, build the model with only its first `keep_layers` decoder blocks.
    The output of block L depends only on blocks 0..L, so the readout is unchanged; this
    lets 7-8B models fit on a 16 GB GPU in bf16. Phase 1 verifies the equivalence.
    """
    import transformers
    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(repo)
    kwargs = {"low_cpu_mem_usage": True, "device_map": "cuda" if torch.cuda.is_available() else "cpu"}
    # Newer transformers renamed torch_dtype -> dtype; an unknown kwarg would silently be
    # written into the config, so pick by version and assert the dtype afterwards.
    major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
    kwargs["dtype" if (major, minor) >= (4, 56) else "torch_dtype"] = DTYPES[dtype]
    if attn != "default":
        kwargs["attn_implementation"] = attn
    if keep_layers is not None:
        cfg = AutoConfig.from_pretrained(repo)
        cfg.num_hidden_layers = keep_layers
        if isinstance(getattr(cfg, "layer_types", None), list):
            cfg.layer_types = cfg.layer_types[:keep_layers]
        kwargs["config"] = cfg
    model = AutoModelForCausalLM.from_pretrained(repo, **kwargs)
    model.eval()
    got = next(model.parameters()).dtype
    assert got == DTYPES[dtype], f"model loaded as {got}, expected {DTYPES[dtype]}"
    return tok, model


def decoder_layers(model):
    return model.model.layers if hasattr(model.model, "layers") else model.model.language_model.layers


def encode(tok, text, device):
    """As the repo does for base models: default special tokens (adds BOS where the tokenizer does)."""
    return tok(text, return_tensors="pt").input_ids.to(device)


class FinalTokenReader:
    """Forward hook on decoder block `layer`; captures the final-token output in fp32."""

    def __init__(self, model, layer):
        self.model, self.act = model, None
        self.handle = decoder_layers(model)[layer].register_forward_hook(self._hook)

    def _hook(self, module, inputs, output):
        hs = output[0] if isinstance(output, tuple) else output
        self.act = hs[0, -1, :].float().cpu().numpy()

    @torch.no_grad()
    def __call__(self, input_ids):
        self.model(input_ids=input_ids)
        return self.act

    def remove(self):
        self.handle.remove()


def git_head(path):
    try:
        return subprocess.run(["git", "-C", str(path), "rev-parse", "HEAD"],
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "unknown"


def env_info():
    import transformers
    info = {"python": sys.version.split()[0], "torch": torch.__version__,
            "transformers": transformers.__version__, "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
    if torch.cuda.is_available():
        info["gpu_capability"] = ".".join(map(str, torch.cuda.get_device_capability(0)))
    return info


def pip_freeze(path):
    out = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    Path(path).write_text(out)


def zscore(x, mean, sd):
    return (np.asarray(x) - mean) / (sd + 1e-8)

In [ ]:
%%writefile /content/speaker_study_scripts/02_build_stimuli.py
"""Phase 2: build the label-swapped stimuli and verify them.

For each of the 420 base-model transcripts, writes three variants that are byte-identical
except for the final speaker label:
  assistant_next   ...\\n[Assistant]:   (original)
  user_next        ...\\n[User]:
  moderator_next   ...\\n[Moderator]:   (third label, fixed before any Phase 3 run)

Text checks (always): each variant equals the shared prefix plus its label, the prefix
ends with '\\n', and the variants differ only after the prefix.

Token checks (with --tokenizers, needs Hugging Face access): tokenizes every variant,
finds the common token prefix across the three conditions, and records the differing
tail tokens and the final token of each condition. The tail may be shorter than the label
(e.g. if '\\n[' is one token shared by all labels); an item is flagged only if its tail
is not a non-empty suffix of the label, i.e. if tokens outside the label differ.

Outputs (never overwritten):
  <out-dir>/stimuli.jsonl                       one row per scenario x condition
  <out-dir>/token_check_<model>.csv / .json     per-tokenizer checks (if --tokenizers)
"""

import argparse
import json
from pathlib import Path

import pa_common as pc

LABELS = {"assistant_next": "[Assistant]:", "user_next": "[User]:", "moderator_next": "[Moderator]:"}
ORIG = LABELS["assistant_next"]


def build(items):
    rows = []
    for it in items:
        text = it["text"]
        assert text.endswith("\n" + ORIG), it["id"]
        prefix = text[: -len(ORIG)]
        for cond, label in LABELS.items():
            v = prefix + label
            assert v[: len(prefix)] == prefix and v[len(prefix):] == label
            rows.append({"scenario_id": it["id"], "category": it["category"], "group": it["group"],
                         "n_user_turns": it["n_user_turns"], "condition": cond, "label": label, "text": v})
        assert rows[-3]["text"] == text  # assistant_next is the original, byte for byte
    return rows


def token_check(rows, repo):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(repo)
    by_id = {}
    for r in rows:
        by_id.setdefault(r["scenario_id"], {})[r["condition"]] = r
    out, problems = [], []
    for sid, conds in by_id.items():
        ids = {c: tok(conds[c]["text"]).input_ids for c in LABELS}
        n = min(len(v) for v in ids.values())
        k = 0
        while k < n and len({tuple(v[: k + 1]) for v in ids.values()}) == 1:
            k += 1
        rec = {"scenario_id": sid, "n_common_tokens": k}
        for c, v in ids.items():
            tail = v[k:]
            rec[f"{c}_n_tokens"] = len(v)
            rec[f"{c}_tail_ids"] = " ".join(map(str, tail))
            rec[f"{c}_tail"] = tok.decode(tail)
            rec[f"{c}_final_id"] = v[-1]
            rec[f"{c}_final"] = tok.decode(v[-1:])
            if not tail or not LABELS[c].endswith(tok.decode(tail)):
                problems.append((sid, c, tok.decode(tail)))
        out.append(rec)
    return out, problems


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--pain-axis-dir", required=True)
    p.add_argument("--out-dir", required=True)
    p.add_argument("--tokenizers", nargs="*", default=[], choices=sorted(pc.MODELS))
    args = p.parse_args()

    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    rows = build(pc.load_scenarios(args.pain_axis_dir))
    stim = out / "stimuli.jsonl"
    if stim.exists():
        existing = [json.loads(line) for line in stim.read_text().splitlines()]
        assert existing == rows, f"{stim} exists with different content; refusing to overwrite"
        print(f"{stim} already exists and matches")
    else:
        stim.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows))
        print(f"wrote {stim}: {len(rows)} rows")

    import pandas as pd
    for repo in args.tokenizers:
        name = pc.MODELS[repo]
        csv_path, json_path = out / f"token_check_{name}.csv", out / f"token_check_{name}.json"
        if csv_path.exists():
            print(f"{csv_path} exists; skipping")
            continue
        recs, problems = token_check(rows, repo)
        df = pd.DataFrame(recs)
        df.to_csv(csv_path, index=False)
        summary = {"model": name, "repo": repo, "n_scenarios": len(df),
                   "n_tail_mismatches": len(problems), "tail_mismatch_examples": problems[:10]}
        for c in LABELS:
            summary[f"{c}_tails"] = df[f"{c}_tail_ids"].value_counts().to_dict()
            summary[f"{c}_final_tokens"] = (df[f"{c}_final"] + " (" + df[f"{c}_final_id"].astype(str) + ")").value_counts().to_dict()
        summary["same_final_token_all_conditions"] = bool(
            (df["assistant_next_final_id"] == df["user_next_final_id"]).all()
            and (df["assistant_next_final_id"] == df["moderator_next_final_id"]).all())
        json_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
        print(json.dumps(summary, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/speaker_study_scripts/05_layer_sweep.py
"""EXPLORATORY layer sweep (not preregistered; plan in DEVIATIONS.md, entries 34-35).

For one model, at EVERY decoder block:
  1. Runs the paper's pain sentences (S1_1P, S1_3P, S2_1P, S2_3P; final token) and rebuilds
     the S1 and S2 pain vectors with the paper's recipe (compute_pain_vector, copied from
     Pain-axis scripts/3.2_pain_vectors/01_extract_activations_and_pain_vectors.py).
     For tokenizers without a BOS token (Qwen), both no prefix and an eos-as-BOS prefix
     (what TransformerLens prepends) are tried; the variant closer to the shipped vectors
     is used. Sentence data only.
  2. Runs all stimuli (stimuli.jsonl) and projects the final-token output of every block
     onto that block's rebuilt unit S1/S2 vectors; at the steering and extraction layers
     also onto the shipped vectors.

Readout convention as in Phases 1-3: forward hook on decoder block L = hidden_states[L+1].
Loading (DEVIATIONS.md, entry 36): the model is built without its LM head (AutoModel), with
only blocks 0..--max-layer, and its weights are loaded straight onto the GPU
(device_map={"": 0}; no CPU offload, no full copy in CPU RAM). If any parameter or buffer
is not on the GPU afterwards, the script stops with an error. No quantization.

Outputs (new folder, never overwritten) <out-root>/results/<model>/layersweep_<run-id>/:
  stimuli_proj.csv   scenario x condition x layer: proj_S1, proj_S2, act_norm (+ shipped)
  sentence_proj.csv  sentence x layer: projections for the §3.3 check
  vectors.npz        rebuilt unit S1/S2 vectors per layer (chosen BOS variant)
  run_info.json      cosines with the shipped vectors, chosen variant, in-sample AUC per layer
"""

import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import pa_common as pc

PAIN_CATEGORIES = ["A1", "A2", "A3", "A4", "A5"]
CONTROL_CATEGORIES = ["B", "C1", "C2", "D", "E"]
DENOISE_VARIANCE = 0.5
SENTENCE_SETS = ["S1_1P", "S1_3P", "S2_1P", "S2_3P"]


def compute_pain_vector(acts, cats):
    """Paper's recipe: pain mean minus all-controls mean, denoised against control PCs (50%)."""
    from sklearn.decomposition import PCA
    cats = np.array(cats)
    pain_mean = acts[np.isin(cats, PAIN_CATEGORIES)].mean(0)
    control = acts[np.isin(cats, CONTROL_CATEGORIES)]
    control_mean = control.mean(0)
    vec = pain_mean - control_mean
    pca = PCA().fit(control - control_mean)
    n = min(int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), DENOISE_VARIANCE)) + 1,
            len(pca.components_))
    for d in pca.components_[:n]:
        vec = vec - np.dot(vec, d) * d
    return vec


def unit(v):
    v = np.asarray(v, dtype=np.float64)
    return v / (np.linalg.norm(v) + 1e-12)


def load_for_sweep(repo, dtype, max_layer=None, allow_cpu=False):
    """Load the base model (no LM head) with blocks 0..max_layer, weights straight to the GPU."""
    import transformers
    from transformers import AutoConfig, AutoModel, AutoTokenizer
    on_gpu = torch.cuda.is_available()
    if not on_gpu and not allow_cpu:
        raise SystemExit("ERROR: no CUDA GPU visible. Set Runtime -> Change runtime type -> T4 GPU, then rerun.")
    tok = AutoTokenizer.from_pretrained(repo)
    cfg = AutoConfig.from_pretrained(repo)
    if max_layer is not None:
        if not 0 <= max_layer < cfg.num_hidden_layers:
            raise SystemExit(f"ERROR: --max-layer {max_layer} outside 0..{cfg.num_hidden_layers - 1}")
        cfg.num_hidden_layers = max_layer + 1
        if isinstance(getattr(cfg, "layer_types", None), list):
            cfg.layer_types = cfg.layer_types[: max_layer + 1]
    kw = {"config": cfg, "low_cpu_mem_usage": True}
    if on_gpu:
        kw["device_map"] = {"": 0}  # every weight straight to cuda:0; nothing offloaded
    major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
    kw["dtype" if (major, minor) >= (4, 56) else "torch_dtype"] = pc.DTYPES[dtype]
    model = AutoModel.from_pretrained(repo, **kw)
    model.eval()
    devices = sorted({str(t.device) for t in list(model.parameters()) + list(model.buffers())})
    if on_gpu and any(not d.startswith("cuda") for d in devices):
        raise SystemExit(f"ERROR: model tensors landed on {devices}, not only on the GPU. Stopping: a CPU-resident "
                         "model would be too slow and can exhaust Colab RAM. Restart the runtime (Runtime -> "
                         "Disconnect and delete runtime) and run again; if it repeats, tell Claude.")
    dt = {str(t.dtype) for t in model.parameters()}
    if dt != {str(pc.DTYPES[dtype])}:
        raise SystemExit(f"ERROR: parameter dtypes {dt}, expected {pc.DTYPES[dtype]}")
    return tok, model, devices


class AllLayerReader:
    """Hooks on every decoder block; captures the final-token output of each, in fp32."""

    def __init__(self, model):
        self.layers = model.layers if hasattr(model, "layers") else model.model.layers
        self.acts = [None] * len(self.layers)
        self.handles = [layer.register_forward_hook(self._hook(i)) for i, layer in enumerate(self.layers)]
        self.model = model
        self.device = next(model.parameters()).device

    def _hook(self, i):
        def f(module, inputs, output):
            hs = output[0] if isinstance(output, tuple) else output
            self.acts[i] = hs[0, -1, :].float().cpu().numpy()
        return f

    @torch.no_grad()
    def __call__(self, ids):
        self.model(input_ids=torch.tensor([ids], device=self.device))
        return np.stack(self.acts)  # (n_layers, d)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--pain-axis-dir", required=True)
    ap.add_argument("--stimuli", required=True)
    ap.add_argument("--out-root", required=True)
    ap.add_argument("--model-repo", default="Qwen/Qwen2.5-7B", choices=sorted(pc.MODELS))
    ap.add_argument("--dtype", default="bf16", choices=sorted(pc.DTYPES))
    ap.add_argument("--max-layer", type=int, default=None, help="last decoder block to load and analyse (default: all)")
    ap.add_argument("--allow-cpu", action="store_true", help="local testing only")
    ap.add_argument("--run-id", default=None)
    args = ap.parse_args()

    np.random.seed(pc.SEED)
    torch.manual_seed(pc.SEED)
    name = pc.MODELS[args.model_repo]
    env = pc.env_info()
    run_id = args.run_id or time.strftime("%Y%m%d_%H%M%S") + "_" + (env["gpu"] or "cpu").replace(" ", "")
    out = Path(args.out_root) / "results" / name / f"layersweep_{run_id}"
    if out.exists():
        raise SystemExit(f"{out} exists; refusing to overwrite")

    pa = Path(args.pain_axis_dir)
    steer_layer, shipped_steer = pc.load_vectors(pa, name)
    pv = torch.load(pa / "results/3.2_pain_vectors/pain_vectors" / name / "pain_vectors.pt", map_location="cpu",
                    weights_only=False)
    extr_layer = int(pv["layer"])
    shipped = {steer_layer: {"S1": shipped_steer["s1_pain_vector"], "S2": shipped_steer["s2_pain_vector"]},
               extr_layer: {"S1": unit(pv["s1_pain_vector"].numpy()), "S2": unit(pv["s2_pain_vector"].numpy())}}

    if args.max_layer is not None and args.max_layer < extr_layer:
        raise SystemExit(f"ERROR: --max-layer {args.max_layer} is below the extraction layer {extr_layer}")
    tok, model, devices = load_for_sweep(args.model_repo, args.dtype, args.max_layer, args.allow_cpu)
    reader = AllLayerReader(model)
    n_layers = len(reader.layers)
    gpu_mem = None
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        gpu_mem = {"allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 2),
                   "free_gib": round(free / 2**30, 2), "total_gib": round(total / 2**30, 2)}
    print(f"{name}: {n_layers} blocks loaded (0..{n_layers - 1}); steering layer {steer_layer}, extraction layer "
          f"{extr_layer}; tensors on {devices}; GPU memory {gpu_mem}")
    out.mkdir(parents=True)  # only after a successful GPU load
    pc.pip_freeze(out / "env.txt")

    # ---- 1. sentences and rebuilt vectors -------------------------------------------
    data = json.load(open(pa / "datasets" / "3.1_pain_and_control_datasets.json"))["datasets"]
    variants = {"hf_default": lambda ids: ids}
    if tok.bos_token_id is None:
        variants["eos_as_bos"] = lambda ids: [tok.eos_token_id] + ids
    sent_acts = {v: {} for v in variants}
    t0 = time.time()
    for v, prefix in variants.items():
        for ds in SENTENCE_SETS:
            sent_acts[v][ds] = np.stack([reader(prefix(tok(s["prompt"]).input_ids)) for s in data[ds]["sentences"]])
        print(f"  sentences, variant {v}: done ({time.time() - t0:.0f}s)")
    cats = {ds: [s["category"] for s in data[ds]["sentences"]] for ds in SENTENCE_SETS}

    def rebuild(v):
        return {L: {"S1": unit(compute_pain_vector(sent_acts[v]["S1_1P"][:, L], cats["S1_1P"])),
                    "S2": unit(compute_pain_vector(sent_acts[v]["S2_1P"][:, L], cats["S2_1P"]))}
                for L in range(n_layers)}

    rebuilt, cosines = {}, {}
    for v in variants:
        rebuilt[v] = rebuild(v)
        cosines[v] = {f"L{L}_{k}": float(np.dot(rebuilt[v][L][k], shipped[L][k])) for L in shipped for k in ("S1", "S2")}
    chosen = max(variants, key=lambda v: np.mean(list(cosines[v].values())))
    vecs = rebuilt[chosen]
    print("cosine with shipped vectors:", json.dumps(cosines), "-> chosen:", chosen)

    from sklearn.metrics import roc_auc_score
    auc = {}
    for L in range(n_layers):
        a = sent_acts[chosen]["S2_1P"][:, L] @ vecs[L]["S2"]
        auc[L] = float(roc_auc_score(np.isin(cats["S2_1P"], PAIN_CATEGORIES), a))

    rows = []
    for ds in SENTENCE_SETS:
        A = sent_acts[chosen][ds]
        for i, s in enumerate(data[ds]["sentences"]):
            for L in range(n_layers):
                rows.append({"set": ds, "idx": i, "category": s["category"], "layer": L,
                             "proj_S1": float(A[i, L] @ vecs[L]["S1"]), "proj_S2": float(A[i, L] @ vecs[L]["S2"])})
    pd.DataFrame(rows).to_csv(out / "sentence_proj.csv", index=False)
    np.savez_compressed(out / "vectors.npz", S1=np.stack([vecs[L]["S1"] for L in range(n_layers)]).astype(np.float32),
                        S2=np.stack([vecs[L]["S2"] for L in range(n_layers)]).astype(np.float32))
    del sent_acts

    # ---- 2. stimuli at every layer ---------------------------------------------------
    stimuli = [json.loads(line) for line in Path(args.stimuli).read_text().splitlines()]
    rows, t0 = [], time.time()
    for j, s in enumerate(stimuli):
        A = reader(tok(s["text"]).input_ids)  # stimuli: HF default, exactly as in Phases 1-3
        for L in range(n_layers):
            r = {"scenario_id": s["scenario_id"], "category": s["category"], "group": s["group"],
                 "n_user_turns": s["n_user_turns"], "condition": s["condition"], "layer": L,
                 "proj_S1": float(A[L] @ vecs[L]["S1"]), "proj_S2": float(A[L] @ vecs[L]["S2"]),
                 "act_norm": float(np.linalg.norm(A[L]))}
            if L in shipped:
                r["proj_S1_shipped"] = float(A[L] @ shipped[L]["S1"])
                r["proj_S2_shipped"] = float(A[L] @ shipped[L]["S2"])
            rows.append(r)
        if (j + 1) % 200 == 0:
            print(f"  stimuli {j + 1}/{len(stimuli)} ({time.time() - t0:.0f}s)")
    pd.DataFrame(rows).to_csv(out / "stimuli_proj.csv", index=False)

    info = {"EXPLORATORY": True, "model": name, "model_repo": args.model_repo, "run_id": run_id, "n_layers": n_layers,
            "steering_layer": steer_layer, "extraction_layer": extr_layer, "dtype": args.dtype, "env": env,
            "max_layer": args.max_layer, "blocks_loaded": n_layers, "tensor_devices": devices, "gpu_memory": gpu_mem,
            "pain_axis_commit": pc.git_head(pa), "bos_variants": list(variants), "chosen_variant": chosen,
            "cosine_with_shipped": cosines, "in_sample_auc_S2_1P": auc}
    (out / "run_info.json").write_text(json.dumps(info, indent=2))
    print(f"wrote {out}")


if __name__ == "__main__":
    main()

In [ ]:
# Stimuli: same frozen file as Phase 3 (hash check).
%cd /content/speaker_study_scripts
!python 02_build_stimuli.py --pain-axis-dir /content/Pain-axis --out-dir "{OUT_ROOT}/stimuli"
import hashlib
got = hashlib.sha256(open(f"{OUT_ROOT}/stimuli/stimuli.jsonl", "rb").read()).hexdigest()
assert got == "9d3b84ca8e32e20d07fea37a2884119739833abe711b0c242e039ed76b03d209", "stimuli.jsonl hash mismatch; do not run, tell Claude"
print("ok stimuli.jsonl", got[:12])

In [ ]:
# GPU pre-flight: the whole model must fit on this GPU.
import torch
assert torch.cuda.is_available(), "ERROR: no GPU. Runtime -> Change runtime type -> T4 GPU."
free, total = torch.cuda.mem_get_info()
print(f"GPU: {torch.cuda.get_device_name(0)}, free {free / 2**30:.1f} of {total / 2**30:.1f} GiB")
assert free / 2**30 > 13.0, "ERROR: less than 13 GiB free on the GPU. Restart the runtime and run again."

In [ ]:
# Reuse a complete Gemma sweep from an earlier run if one is on Drive (run_info.json is written last).
import glob, json, os
done = [d for d in sorted(glob.glob(f"{OUT_ROOT}/results/Gemma_2_2B_base/layersweep_*"))
        if os.path.exists(f"{d}/run_info.json") and os.path.exists(f"{d}/stimuli_proj.csv")]
REUSE = done[-1] if done else None
if REUSE:
    info = json.load(open(f"{REUSE}/run_info.json"))
    print("Found a complete sweep; it will be reused, not rerun:", REUSE)
    print("blocks:", info.get("n_layers"), "| devices:", info.get("tensor_devices", info.get("device_map")))
else:
    print("No complete Gemma sweep on Drive; running it now.")

In [ ]:
# Layer sweep: Gemma 2 2B base (skipped if a complete run was found above)
%cd /content/speaker_study_scripts
if not REUSE:
    !python 05_layer_sweep.py --pain-axis-dir /content/Pain-axis --stimuli "{OUT_ROOT}/stimuli/stimuli.jsonl" --out-root "{OUT_ROOT}" --model-repo google/gemma-2-2b --dtype bf16

In [ ]:
# Bundle the latest complete sweep for Claude.
import glob, os, zipfile
runs = [d for d in sorted(glob.glob(f"{OUT_ROOT}/results/Gemma_2_2B_base/layersweep_*")) if os.path.exists(f"{d}/run_info.json")]
assert runs, "ERROR: no complete sweep folder found (run_info.json missing); see the output of the cell above."
paths = [p for p in glob.glob(runs[-1] + "/*") if os.path.isfile(p)]
zip_path = f"{OUT_ROOT}/speaker_study_layersweep_gemma.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p, os.path.relpath(p, OUT_ROOT))
print("\n".join(os.path.relpath(p, OUT_ROOT) for p in paths))
from google.colab import files
files.download(zip_path)